# NIFTY 50 Regime Change Pipeline

Run the project scripts in order: Phase 1 builds a local NIFTY 50 time-series snapshot and causal features; Phase 2 runs as-of PELT–L2 and PELT–RBF detection using that snapshot.

**Look-ahead controls:** Phase 1 features for date *t* use closes only through the previous observed session. Splits are chronological. Phase 2 fits calibration on training data and excludes the test period unless explicitly enabled after model choices are frozen.

Keep each output directory new and empty. The scripts refuse to overwrite nonempty folders.

## 0. Install dependencies

Run once in the active Jupyter environment. Restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -r requirements.txt

## 1. Set paths and data source

Keep `phase1_nifty.py`, `phase2_cpd.py`, and this notebook in the same project folder. Use `yahoo` to download NIFTY 50 through yfinance, or set `DATA_SOURCE = "local_csv"` and provide a folder containing `NIFTY50.csv` with `Date,Close` columns. The local CSV should contain enough history for the chronological train, validation, and test periods.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path.cwd()  # change this if the notebook is not in the project folder
PHASE1_OUTPUT = PROJECT_DIR / 'phase1_nifty50'
PHASE2_OUTPUT = PROJECT_DIR / 'phase2_nifty50_validation'
DATA_SOURCE = 'yahoo'  # choose 'yahoo' or 'local_csv'
LOCAL_CSV_DIR = PROJECT_DIR / 'index_csvs'  # used only for local_csv

assert (PROJECT_DIR / 'phase1_nifty.py').exists(), 'phase1_nifty.py must be in PROJECT_DIR'
assert (PROJECT_DIR / 'phase2_cpd.py').exists(), 'phase2_cpd.py must be in PROJECT_DIR'
print('Project folder:', PROJECT_DIR)
print('Data source:', DATA_SOURCE)

## 2. Run Phase 1 checks

These synthetic self-tests verify the script without downloading market data.

In [ ]:
import phase1_nifty
phase1_nifty.main(['--self-test'])

## 3. Build the NIFTY 50 time series and features

This creates raw closes, aligned closes, feature rows, chronological splits, plots, and a manifest with hashes. The end date defaults to the latest completed date available to the script. Choose a new output folder for each snapshot.

In [ ]:
phase1_args = ['--indices', 'NIFTY50', '--output', str(PHASE1_OUTPUT)]
if DATA_SOURCE == 'local_csv':
    phase1_args += ['--input-dir', str(LOCAL_CSV_DIR)]
elif DATA_SOURCE != 'yahoo':
    raise ValueError("DATA_SOURCE must be 'yahoo' or 'local_csv'")

phase1_nifty.main(phase1_args)

## 4. Inspect the Phase 1 snapshot

Check that the manifest exists and that the feature file has complete rows before proceeding.

In [ ]:
manifest_path = PHASE1_OUTPUT / 'manifest.json'
features_path = PHASE1_OUTPUT / 'features' / 'NIFTY50_features.csv'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
features = pd.read_csv(features_path, parse_dates=['Date', 'source_date'])
print('Feature rows:', len(features))
print('Date range:', features['Date'].min(), 'to', features['Date'].max())
print('Rows by split:')
display(features['split'].value_counts(dropna=False).rename_axis('split').to_frame('rows'))
print('Complete rows:', int(features['complete'].fillna(False).astype(bool).sum()))
print('Manifest source:', manifest.get('source'))
display(features.head())

## 5. Run Phase 2 checks

This verifies sequential detection behavior on synthetic series. It does not use your market data.

In [ ]:
import phase2_cpd
phase2_cpd.main(['--self-test'])

## 6. Run as-of change-point detection

The default uses validation-period data and leaves test data closed. The decision log records when an alert could have been issued; the estimated break date is kept separately. Each rerun needs a new empty output directory.

In [ ]:
phase2_args = [
    '--input', str(PHASE1_OUTPUT),
    '--indices', 'NIFTY50',
    '--output', str(PHASE2_OUTPUT),
]
phase2_cpd.main(phase2_args)

## 7. Review Phase 2 outputs

The output folder contains decision logs, confirmed event logs, calibration details, timeline plots, a summary, and a run manifest. Review validation results before changing settings. Do not use test results to tune parameters.

In [ ]:
summary_path = PHASE2_OUTPUT / 'summary.csv'
summary = pd.read_csv(summary_path)
display(summary)
print('Output files:')
for path in sorted(PHASE2_OUTPUT.iterdir()):
    print(path.name)

## Optional: final test-period evaluation

Only after you have selected and frozen the features, penalties, window, and alert rules using training/validation data, run Phase 2 once with `--include-test` and a fresh output folder. Do not select model settings based on this final result.

```python
FINAL_OUTPUT = PROJECT_DIR / 'phase2_nifty50_final'
phase2_cpd.main([
    '--input', str(PHASE1_OUTPUT),
    '--indices', 'NIFTY50',
    '--output', str(FINAL_OUTPUT),
    '--include-test',
])
```